In [4]:
import numpy as np
import pandas as pd
import itertools
from collections import defaultdict
from typing import Dict, Tuple, Any, List
from scipy import stats
from statsmodels.stats.power import TTestIndPower
import warnings
import os

try:
    from tqdm.auto import tqdm  
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else range(0)


os.chdir('../../..')

from Scripts.Spectral_Analysis.Spectrum_Filter import filter_spectras
warnings.filterwarnings("ignore")

In [16]:
FEATURE_GROUPS = {
    "day_time": {
        "Day":      ["Day"],
        "Evening":  ["Evening"],
    },

    "stim_type": {
        "r": ["r"],
        "g": ["g"],
    },

    "stim_label": {
        "line":   [0, 1, 4, 6, 7, 11, 8],
        "figure": [2, 3, 5, 9, 10, 12],
    },

    "gender": {
        "m": ["m"],
        "f": ["f"],
    },

    "age": {
        "18-22": [18, 19, 20, 21, 22],
        "23-29": [23, 24, 25, 25, 26, 27, 28, 29],
        "30-35": [30, 31, 32, 33, 34, 35],
    },

    "handiness": {
        "l": ["l"],
        "r": ["r"],
    },
}


DEFAULT_BANDS = {
    'Delta': (1, 4),
    'Tetta': (4, 7),
    'Alpha': (7, 13),
    'Beta': (13, 30)
}

_RESULT_FIELDS = [
    "power",
    "phase",
    "s_id",
    "t_id",
    "gender",
    "handiness",
    "age",
    "label",
    "img",
    "task_type",
    "day_time",
    "stim_type",
]


def normalize_results(raw_results: List[Any]) -> List[Dict[str, Any]]:
    """
    Превращает вывод filter_spectras(...) в список словарей.
    Если каких-то полей нет в конкретном npz — просто не добавляем.
    Важно: нам критично только наличие 'power' и 's_id'.
    """
    norm = []
    for item in raw_results:
        if isinstance(item, dict):
            norm.append(item.copy())
        else:
            rec = {}
            for idx, key in enumerate(_RESULT_FIELDS):
                if idx < len(item):
                    rec[key] = item[idx]
            norm.append(rec)
    return norm

def make_band_cols_from_bands(
    bands: Dict[str, Tuple[float, float]],
    freqs: np.ndarray,
) -> Dict[str, np.ndarray]:
    """
    bands: {"alpha": (8,12), ...} в Гц
    freqs: массив частот для этого npz (длины n_freqs)
    Возвращает:
        {"alpha": array([...индексы...]), "beta": array([...]), ...}
    """
    band_cols = {}
    for band_name, (f_lo, f_hi) in bands.items():
        idx = np.where((freqs >= f_lo) & (freqs <= f_hi))[0]
        band_cols[band_name] = idx
    return band_cols

def aggregate_by_subject(
    results: List[Dict[str, Any]],
    band_cols: Dict[str, np.ndarray],
    n_channels: int,
) -> Dict[Tuple[int, str], Dict[str, Any]]:
    """
    ВЕРСИЯ ПО-ТРИАЛАМ (без усреднения по субъектам).
    Накапливает по каждому (канал, бэнд) значения всех триалов как независимые наблюдения.

    Возвращает:
        trial_vecs[(ch, band)] = {
            'ids'   : List[trial_id или None],   # для совместимости с прежним кодом
            's_ids' : List[subject_id или None], # полезно иметь под рукой
            'vals'  : np.ndarray[float],         # одно значение на триал
        }
    """
    trial_vecs: Dict[Tuple[int, str], Dict[str, Any]] = defaultdict(
        lambda: {'ids': [], 's_ids': [], 'vals': []}
    )

    for rec in results:
        power = rec.get("power", None)
        s_id  = rec.get("s_id", None)
        t_id  = rec.get("t_id", None)

        if power is None:
            continue  # без спектра ничего не сделаем

        # (C,F,T) -> среднее по времени, иначе ожидаем (C,F)
        if hasattr(power, "ndim") and power.ndim == 3:
            vecs = power.mean(axis=2).astype(np.float32, copy=False)
        else:
            vecs = np.asarray(power, dtype=np.float32)

        if vecs.ndim != 2:
            continue  # ожидаем (C,F)

        # привести идентификаторы к скалярам, если возможно
        try:
            sid_int = int(s_id) if s_id is not None else None
        except Exception:
            sid_int = None
        try:
            tid_int = int(t_id) if t_id is not None else None
        except Exception:
            tid_int = None

        # обходим каналы и бэнды
        max_ch = min(n_channels, vecs.shape[0])
        for ch in range(max_ch):
            for band_name, freq_idx in band_cols.items():
                if freq_idx.size == 0 or np.max(freq_idx) >= vecs.shape[1]:
                    continue
                band_val = float(np.nanmean(vecs[ch, freq_idx]))

                key = (ch, band_name)
                trial_vecs[key]['ids'].append(tid_int)     # trial id (может быть None)
                trial_vecs[key]['s_ids'].append(sid_int)   # subject id (может быть None)
                trial_vecs[key]['vals'].append(band_val)

    # списки → numpy
    for key, d in trial_vecs.items():
        d['vals'] = np.asarray(d['vals'], dtype=float)

    return trial_vecs


def build_two_condition_vectors(
    condA_subj: Dict[Tuple[int, str], Dict[str, Any]],
    condB_subj: Dict[Tuple[int, str], Dict[str, Any]],
    condA_name: str,
    condB_name: str,
    n_channels: int,
    band_names: List[str],
) -> Dict[Tuple[int, str], Dict[str, Any]]:
    """
    Сводит по-триальные векторы двух условий в единую структуру.
    На вход подаются результаты aggregate_by_subject (теперь это по-триально).

    Возвращает:
        {(ch, band): {
            condA_name            : np.ndarray[float],  # все триалы A
            condB_name            : np.ndarray[float],  # все триалы B
            f"{condA_name}_ids"   : List[trial_id],
            f"{condB_name}_ids"   : List[trial_id],
            f"{condA_name}_s_ids" : List[subject_id],
            f"{condB_name}_s_ids" : List[subject_id],
        }}
    """
    subject_vectors: Dict[Tuple[int, str], Dict[str, Any]] = {}

    for ch in range(n_channels):
        for band_name in band_names:
            key = (ch, band_name)

            A = condA_subj.get(key, {'ids': [], 's_ids': [], 'vals': np.array([], float)})
            B = condB_subj.get(key, {'ids': [], 's_ids': [], 'vals': np.array([], float)})

            x = np.asarray(A.get('vals', np.array([], float)), dtype=float)
            y = np.asarray(B.get('vals', np.array([], float)), dtype=float)

            subject_vectors[key] = {
                condA_name: x,
                condB_name: y,
                f"{condA_name}_ids": list(A.get('ids', [])),
                f"{condB_name}_ids": list(B.get('ids', [])),
                f"{condA_name}_s_ids": list(A.get('s_ids', [])),
                f"{condB_name}_s_ids": list(B.get('s_ids', [])),
            }

    return subject_vectors

def hedges_g(x, y) -> float:
    """
    Hedges' g для двух независимых групп (с поправкой J).
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan

    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)

    # обе дисперсии нулевые → считаем эффект 0 (со знаком разницы средних)
    if vx == 0 and vy == 0:
        return float(np.sign(np.nanmean(x) - np.nanmean(y)) * 0.0)

    sp2 = ((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2)
    if sp2 <= 0:
        return np.nan

    d = (np.nanmean(x) - np.nanmean(y)) / np.sqrt(sp2)
    J = 1 - 3 / (4 * (nx + ny) - 9) if (nx + ny) > 2 else 1.0
    return float(J * d)


def build_band_table_two_conditions(
    subject_vectors: Dict[Tuple[int, str], Dict[str, Any]],
    condA_name: str,
    condB_name: str,
    alpha: float = 0.05
) -> pd.DataFrame:
    """
    Для каждой пары (канал, бэнд):
      - кластер-устойчивый t-тест по испытуемому (если можно)
      - иначе Welch t-test
      - p-value, Hedges' g, оценка мощности
    """
    try:
        import statsmodels.api as sm
        _HAS_SM = True
    except Exception:
        _HAS_SM = False

    power_calc = TTestIndPower()
    rows = []

    for (ch, band_name), vecs in subject_vectors.items():
        x = np.asarray(vecs.get(condA_name, []), float)
        y = np.asarray(vecs.get(condB_name, []), float)
        nA, nB = len(x), len(y)
        if nA == 0 or nB == 0:
            continue

        # s_ids для кластеризации (могут отсутствовать)
        sA = np.asarray(vecs.get(f"{condA_name}_s_ids", []))
        sB = np.asarray(vecs.get(f"{condB_name}_s_ids", []))

        # Welch by default
        try:
            t_stat, p_val = stats.ttest_ind(x, y, equal_var=False, nan_policy='omit')
        except Exception:
            t_stat, p_val = np.nan, np.nan

        g = hedges_g(x, y)

        used_cluster = False
        if _HAS_SM and (sA.size == nA) and (sB.size == nB):
            mA = pd.notna(sA); mB = pd.notna(sB)
            x2 = x[mA]; y2 = y[mB]
            sA2 = sA[mA]; sB2 = sB[mB]
            if (len(x2) >= 2 and len(y2) >= 2 and
                len(np.unique(sA2)) >= 2 and len(np.unique(sB2)) >= 2):
                try:
                    y_all = np.concatenate([x2, y2])
                    g_all = np.concatenate([np.zeros_like(x2), np.ones_like(y2)])
                    sid_all = np.concatenate([sA2, sB2])
                    X = sm.add_constant(g_all.astype(float))
                    res = sm.OLS(y_all, X, missing='drop').fit(
                        cov_type='cluster', cov_kwds={'groups': sid_all}
                    )
                    t_stat = float(res.tvalues[1])
                    p_val  = float(res.pvalues[1])
                    used_cluster = True

                    dfA = pd.DataFrame({'sid': sA2, 'val': x2}).groupby('sid')['val'].mean()
                    dfB = pd.DataFrame({'sid': sB2, 'val': y2}).groupby('sid')['val'].mean()
                    nA_eff, nB_eff = len(dfA), len(dfB)
                    if nA_eff >= 2 and nB_eff >= 2:
                        g_eff = hedges_g(dfA.values, dfB.values)
                        ratio = nB_eff / max(nA_eff, 1)
                        power = power_calc.solve_power(effect_size=abs(g_eff),
                                                       nobs1=nA_eff, ratio=ratio,
                                                       alpha=alpha, alternative='two-sided')
                    else:
                        power = np.nan
                except Exception:
                    used_cluster = False

        if not used_cluster:
            try:
                ratio = nB / max(nA, 1)
                power = power_calc.solve_power(effect_size=abs(g),
                                               nobs1=nA, ratio=ratio,
                                               alpha=alpha, alternative='two-sided')
            except Exception:
                power = np.nan

        rows.append({
            'channel': ch,
            'band': band_name,
            f'n_{condA_name}': nA,
            f'n_{condB_name}': nB,
            f'mean_{condA_name}': float(np.nanmean(x)) if nA else np.nan,
            f'mean_{condB_name}': float(np.nanmean(y)) if nB else np.nan,
            f'delta_{condA_name}_minus_{condB_name}':
                float(np.nanmean(x) - np.nanmean(y)) if (nA and nB) else np.nan,
            't_stat': float(t_stat) if np.isfinite(t_stat) else np.nan,
            'p_value': float(p_val) if np.isfinite(p_val) else np.nan,
            'hedges_g': float(g) if np.isfinite(g) else np.nan,
            'power': float(power) if np.isfinite(power) else np.nan,
            'sig_alpha_0.05': bool((p_val <= 0.05) if np.isfinite(p_val) else False),
            'sig_alpha_0.01': bool((p_val <= 0.01) if np.isfinite(p_val) else False),
            'sig_and_power': bool((p_val <= 0.05) and (power >= 0.8))
                              if (np.isfinite(p_val) and np.isfinite(power)) else False,
            'clustered': used_cluster,
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['p_value', 'power'], ascending=[True, True]).reset_index(drop=True)
    return df



def run_stats_between_groups(
    data_A_raw,
    data_B_raw,
    group_A_name: str,
    group_B_name: str,
    band_cols: Dict[str, np.ndarray],
    n_channels: int,
    alpha: float = 0.05,
):
    # нормализация формата
    data_A = normalize_results(data_A_raw)
    data_B = normalize_results(data_B_raw)

    # агрегация по субъектам
    A_by_subj = aggregate_by_subject(
        data_A,
        band_cols=band_cols,
        n_channels=n_channels,
    )
    B_by_subj = aggregate_by_subject(
        data_B,
        band_cols=band_cols,
        n_channels=n_channels,
    )

    # какие бэнды считаем
    band_names = list(band_cols.keys())

    # векторизуем A и B
    subj_vecs = build_two_condition_vectors(
        condA_subj=A_by_subj,
        condB_subj=B_by_subj,
        condA_name=group_A_name,
        condB_name=group_B_name,
        n_channels=n_channels,
        band_names=band_names,
    )

    # считаем t-тесты и эффекты
    stats_table = build_band_table_two_conditions(
        subject_vectors=subj_vecs,
        condA_name=group_A_name,
        condB_name=group_B_name,
        alpha=alpha,
    )

    return {
        'summary_table': stats_table,
        'meta': {
            'group_A': group_A_name,
            'group_B': group_B_name,
        }
    }


def _collapse_allowed_values(allowed_values: List[Any]):
    """
    Если группа — это одна категория (например ["m"]), вернуть "m".
    Если это бин (например [18,19,20,...]) — вернуть весь список.
    """
    if len(allowed_values) == 1:
        return allowed_values[0]
    return allowed_values

def _has_feature_in_trials(trials: List[Dict[str, Any]], feature_name: str) -> bool:
    for tr in trials:
        if isinstance(tr, dict) and (feature_name in tr) and (tr[feature_name] is not None):
            return True
    return False


def filter_group(
    npz_path: str,
    feature_name: str,
    allowed_values: List[Any],
    extra_kwargs: dict,
):
    # 0) если есть кэш и в нём реально присутствует нужное поле → режем в памяти
    all_trials = extra_kwargs.get("_all_trials")
    if (all_trials is not None) and _has_feature_in_trials(all_trials, feature_name):
        allowed = set(allowed_values)
        out = []
        for tr in all_trials:
            val = tr.get(feature_name, None)
            if val is None:
                continue
            if isinstance(val, (list, tuple, np.ndarray)):
                if any(v in allowed for v in val):
                    out.append(tr)
            else:
                if val in allowed:
                    out.append(tr)
        return out

    # 1) иначе — фолбэк: фильтруем напрямую через filter_spectras (обязательно с метаданными)
    kwargs = {"exec_spec_path": npz_path}
    meta_path = extra_kwargs.get("day_time_meta_path")
    if meta_path is None:
        raise RuntimeError(
            "day_time_meta_path обязателен: передай его в extra_kwargs при вызове run_full_analysis_for_npz(...)"
        )
    kwargs["day_time_meta_path"] = meta_path

    # ["m"] -> "m", ["Day"] -> "Day", [18,19,20] остаётся списком
    val = _collapse_allowed_values(allowed_values)

    if feature_name == "day_time":
        kwargs["day_time"] = val
    elif feature_name == "stim_type":
        kwargs["stim_type"] = val
    elif feature_name == "stim_label":
        kwargs["stim_label"] = val
    elif feature_name == "gender":
        kwargs["gender"] = val
    elif feature_name == "age":
        kwargs["age"] = val
    elif feature_name == "handiness":
        kwargs["handiness"] = val
    else:
        raise ValueError(f"Unknown feature: {feature_name}")

    return filter_spectras(**kwargs)


def prepare_session_config_for_npz(
    npz_path: str,
    bands: Dict[str, Tuple[float, float]] = None,
    extra_kwargs: dict = None,
):

    if bands is None:
        bands = DEFAULT_BANDS.copy()
    if extra_kwargs is None:
        extra_kwargs = {}

    meta_path = extra_kwargs.get("day_time_meta_path", None)
    if meta_path is None:
        raise RuntimeError(
            "day_time_meta_path обязателен: передай extra_kwargs={'day_time_meta_path': '...xlsx'}"
        )

    base_kwargs = {
        "exec_spec_path": npz_path,
        "day_time_meta_path": meta_path,
    }

    all_trials_raw = filter_spectras(**base_kwargs)
    all_trials_norm = normalize_results(all_trials_raw)

    extra_kwargs = dict(extra_kwargs)          # на всякий случай делаем копию
    extra_kwargs["_all_trials"] = all_trials_norm

    # найдём хотя бы один trial с power
    example_power = None
    for rec in all_trials_norm:
        if "power" in rec and rec["power"] is not None:
            example_power = rec["power"]
            break

    if example_power is None:
        raise RuntimeError("Не удалось найти ни одного trial с 'power' в этом npz (после filter_spectras).")

    # определяем размерности
    if hasattr(example_power, "ndim") and example_power.ndim == 3:
        n_channels, n_freqs, _ = example_power.shape
    elif hasattr(example_power, "ndim") and example_power.ndim == 2:
        n_channels, n_freqs = example_power.shape
    else:
        raise ValueError(f"Неожиданная форма 'power': {getattr(example_power,'shape',None)}")

    freqs = np.linspace(2, 40, n_freqs)

    # индексы частот для каждого EEG-бэнда
    band_cols = make_band_cols_from_bands(bands, freqs)

    session_cfg = {
        "npz_path": npz_path,
        "n_channels": n_channels,
        "band_cols": band_cols,
        "bands": bands,
        "extra_kwargs": extra_kwargs,  # важно: тут живёт day_time_meta_path
        "_all_trials": all_trials_norm,   # кэш всех триалов (уже нормализованных)
    }
    return session_cfg

def analyze_single_feature(
    session_cfg: dict,
    feature_name: str,
    alpha: float = 0.05,
):
    """
    Не требуем, чтобы фича уже была в _all_trials.
    Если её нет — filter_group сам сделает fallback через filter_spectras(...) + Excel.
    """
    npz_path    = session_cfg["npz_path"]
    band_cols   = session_cfg["band_cols"]
    n_channels  = session_cfg["n_channels"]
    extra_kwargs= session_cfg["extra_kwargs"]

    group_spec = FEATURE_GROUPS[feature_name]

    # Забираем данные для каждой группы; отбрасываем пустые
    subsets = {}
    for group_label, allowed_values in group_spec.items():
        data = filter_group(
            npz_path=npz_path,
            feature_name=feature_name,
            allowed_values=allowed_values,
            extra_kwargs=extra_kwargs,
        )
        if data and len(data) > 0:
            subsets[group_label] = data

    if len(subsets) < 2:
        return {}

    results_for_feature = {}
    for g1, g2 in itertools.combinations(subsets.keys(), 2):
        data_A_raw = subsets[g1]
        data_B_raw = subsets[g2]
        if not data_A_raw or not data_B_raw:
            continue

        stats_result = run_stats_between_groups(
            data_A_raw=data_A_raw,
            data_B_raw=data_B_raw,
            group_A_name=g1,
            group_B_name=g2,
            band_cols=band_cols,
            n_channels=n_channels,
            alpha=alpha,
        )
        df = stats_result.get("summary_table")
        if df is not None and not df.empty:
            results_for_feature[(g1, g2)] = stats_result

    return results_for_feature

def analyze_feature_pair(
    session_cfg: dict,
    feature_A: str,
    feature_B: str,
    alpha: float = 0.05,
):
    """
    Не требуем наличия фич в _all_trials — filter_group сделает fallback.
    """
    npz_path    = session_cfg["npz_path"]
    band_cols   = session_cfg["band_cols"]
    n_channels  = session_cfg["n_channels"]
    extra_kwargs= session_cfg["extra_kwargs"]

    groups_A = FEATURE_GROUPS[feature_A]
    groups_B = FEATURE_GROUPS[feature_B]

    subsets_A, subsets_B = {}, {}

    for label_A, allowed_values_A in groups_A.items():
        dataA = filter_group(npz_path, feature_A, allowed_values_A, extra_kwargs)
        if dataA and len(dataA) > 0:
            subsets_A[label_A] = dataA

    for label_B, allowed_values_B in groups_B.items():
        dataB = filter_group(npz_path, feature_B, allowed_values_B, extra_kwargs)
        if dataB and len(dataB) > 0:
            subsets_B[label_B] = dataB

    if len(subsets_A) == 0 or len(subsets_B) == 0:
        return {}

    results_for_pair = {}
    for label_A, data_A_raw in subsets_A.items():
        for label_B, data_B_raw in subsets_B.items():
            if not data_A_raw or not data_B_raw:
                continue
            stats_result = run_stats_between_groups(
                data_A_raw=data_A_raw,
                data_B_raw=data_B_raw,
                group_A_name=label_A,
                group_B_name=label_B,
                band_cols=band_cols,
                n_channels=n_channels,
                alpha=alpha,
            )
            df = stats_result.get("summary_table")
            if df is not None and not df.empty:
                results_for_pair[(label_A, label_B)] = stats_result

    return results_for_pair




def run_full_analysis_for_npz(
    npz_path: str,
    alpha: float = 0.05,
    feature_list_single=None,
    feature_pairs=None,
    extra_kwargs: dict = None,
    bands: Dict[str, Tuple[float, float]] = None,
):


    if extra_kwargs is None or "day_time_meta_path" not in extra_kwargs:
        raise RuntimeError(
            "run_full_analysis_for_npz: передай extra_kwargs={'day_time_meta_path': '...xlsx'}"
        )

    session_cfg = prepare_session_config_for_npz(
        npz_path=npz_path,
        bands=bands,
        extra_kwargs=extra_kwargs,
    )

    if feature_list_single is None:
        feature_list_single = list(FEATURE_GROUPS.keys())

    if feature_pairs is None:
        feature_pairs = list(itertools.combinations(FEATURE_GROUPS.keys(), 2))

    final_results = {
        "single_feature": {},
        "pair_feature": {},
    }

    # одиночные фичи (зелёные сравнения)
    for feat in feature_list_single:
        final_results["single_feature"][feat] = analyze_single_feature(
            session_cfg=session_cfg,
            feature_name=feat,
            alpha=alpha,
        )

    # пары фичей (красные сравнения)
    for (featA, featB) in feature_pairs:
        final_results["pair_feature"][(featA, featB)] = analyze_feature_pair(
            session_cfg=session_cfg,
            feature_A=featA,
            feature_B=featB,
            alpha=alpha,
        )

    return final_results



In [6]:
import re, numpy as np, pandas as pd

def probe_npz(npz_path, max_preview=6):
    """
    Печатает ключи, пытается угадать схему хранения спектров и метаданных.
    """
    with np.load(npz_path, allow_pickle=True, mmap_mode="r") as z:
        keys = sorted(list(z.keys()))
        print(f"Файл: {npz_path}")
        print(f"Всего ключей: {len(keys)}")
        print("Первые ключи:", keys[:min(len(keys), 20)])
        
        # кандидаты на спектры
        pow_like = [k for k in keys if re.search(r"(psd|power|spectra|spectrum|fft|spec)", k, re.I)]
        print("\nКлючи, похожие на спектры:", pow_like[:20])

        # есть ли пер-триальная схема (psd_0, psd_1, ...)
        per_trial_idx = []
        for k in pow_like:
            m = re.match(r"^([A-Za-z]+)_(\d+)$", k)
            if m:
                per_trial_idx.append(int(m.group(2)))
        if per_trial_idx:
            print(f"Обнаружен паттерн 'по триалам' вида name_i, индексов: {len(per_trial_idx)}, "
                  f"диапазон {min(per_trial_idx)}..{max(per_trial_idx)}")
            sample_key = [k for k in pow_like if k.endswith(f"_{min(per_trial_idx)}")][0]
            arr = z[sample_key]
            print(f"Пример {sample_key}: shape={arr.shape}, dtype={arr.dtype}")
        else:
            # стековая схема: один ключ с массивом
            candidate = None
            for name in ["psd", "power", "spectra", "spectrum", "PSD", "Power", "X"]:
                if name in z:
                    candidate = name
                    break
            if candidate is None and pow_like:
                candidate = pow_like[0]
            if candidate is not None:
                arr = z[candidate]
                print(f"Похоже, стек: ключ '{candidate}' shape={arr.shape}, dtype={arr.dtype}")
                # попробуем найти мета-массивы той же длины по одной из осей
                meta_arrays = {}
                for meta_name in ["subject_id", "s_id", "sid", "trial_id", "t_id",
                                  "gender", "handiness", "age", "label", "stim_type", "img", "task_type"]:
                    # прямой ключ
                    if meta_name in z:
                        meta_arrays[meta_name] = z[meta_name]
                if meta_arrays:
                    print("Найдены возможные метамассивы:", {k: np.shape(v) for k,v in meta_arrays.items()})
            else:
                print("Не нашёл явного спектрального массива. Нужна ручная проверка.")

        # покажем несколько мета-ключей, если они постфиксные
        meta_postfix = [k for k in keys if re.search(r"(subject|trial|gender|handiness|age|label|stim|task)", k, re.I)]
        print("\nПримеры мета-ключей:", meta_postfix[:min(len(meta_postfix), 20)])


In [7]:
probe_npz(r'./Generated/Spectrums/psds_array_morlet.npz')


Файл: ./Generated/Spectrums/psds_array_morlet.npz
Всего ключей: 186
Первые ключи: ['age_0', 'age_1', 'age_10', 'age_11', 'age_12', 'age_13', 'age_14', 'age_15', 'age_16', 'age_17', 'age_18', 'age_19', 'age_2', 'age_20', 'age_21', 'age_22', 'age_23', 'age_24', 'age_25', 'age_26']

Ключи, похожие на спектры: ['psd_0', 'psd_1', 'psd_10', 'psd_11', 'psd_12', 'psd_13', 'psd_14', 'psd_15', 'psd_16', 'psd_17', 'psd_18', 'psd_19', 'psd_2', 'psd_20', 'psd_21', 'psd_22', 'psd_23', 'psd_24', 'psd_25', 'psd_26']
Обнаружен паттерн 'по триалам' вида name_i, индексов: 31, диапазон 0..30
Пример psd_0: shape=(63, 80), dtype=float64

Примеры мета-ключей: ['age_0', 'age_1', 'age_10', 'age_11', 'age_12', 'age_13', 'age_14', 'age_15', 'age_16', 'age_17', 'age_18', 'age_19', 'age_2', 'age_20', 'age_21', 'age_22', 'age_23', 'age_24', 'age_25', 'age_26']


In [19]:
# ==== imports ====
from typing import List, Any, Dict, Tuple
from collections import defaultdict
import itertools
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.power import TTestIndPower
from contextlib import contextmanager

# ==== helper: временно гасим numpy overflow/invalid ====
@contextmanager
def _np_ignore_overflow():
    """
    Временно игнорируем overflow/invalid/divide в numpy,
    чтобы вызов filter_spectras не падал на «битых» excel-датах и т.п.
    """
    old = np.seterr()
    np.seterr(over='ignore', invalid='ignore', divide='ignore')
    try:
        yield
    finally:
        np.seterr(**old)

# ==== конфигурация фич ====
FEATURE_GROUPS = {
    "day_time": {
        "Day":      ["Day"],
        "Evening":  ["Evening"],
    },

    "stim_type": {
        "r": ["r"],
        "g": ["g"],
    },

    "stim_label": {
        "line":   [0, 1, 4, 6, 7, 11, 8],
        "figure": [2, 3, 5, 9, 10, 12],
    },

    "gender": {
        "m": ["m"],
        "f": ["f"],
    },

    "age": {
        "18-22": [18, 19, 20, 21, 22],
        "23-29": [23, 24, 25, 25, 26, 27, 28, 29],
        "30-35": [30, 31, 32, 33, 34, 35],
    },

    "handiness": {
        "l": ["l"],
        "r": ["r"],
    },
}

DEFAULT_BANDS = {
    'Delta': (1, 4),
    'Tetta': (4, 7),
    'Alpha': (7, 13),
    'Beta': (13, 30)
}

_RESULT_FIELDS = [
    "power",
    "phase",
    "s_id",
    "t_id",
    "gender",
    "handiness",
    "age",
    "label",
    "img",
    "task_type",
    "day_time",
    "stim_type",
]

# ==== utils ====
def normalize_results(raw_results: List[Any]) -> List[Dict[str, Any]]:
    """
    Превращает вывод filter_spectras(...) в список словарей.
    Если каких-то полей нет — просто не добавляем.
    Критично наличие 'power' (и желательно 's_id'/'t_id').
    """
    norm = []
    for item in raw_results:
        if isinstance(item, dict):
            norm.append(item.copy())
        else:
            rec = {}
            for idx, key in enumerate(_RESULT_FIELDS):
                if idx < len(item):
                    rec[key] = item[idx]
            norm.append(rec)
    return norm

def make_band_cols_from_bands(
    bands: Dict[str, Tuple[float, float]],
    freqs: np.ndarray,
) -> Dict[str, np.ndarray]:
    """
    bands: {"alpha": (8,12), ...} в Гц
    freqs: массив частот длины n_freqs
    Возвращает {"alpha": idx_array, ...} — индексы частот для каждого бэнда.
    """
    band_cols = {}
    for band_name, (f_lo, f_hi) in bands.items():
        idx = np.where((freqs >= f_lo) & (freqs <= f_hi))[0]
        band_cols[band_name] = idx
    return band_cols

# ==== агрегация по триалам (каждый триал — независимая точка) ====
def aggregate_by_subject(
    results: List[Dict[str, Any]],
    band_cols: Dict[str, np.ndarray],
    n_channels: int,
) -> Dict[Tuple[int, str], Dict[str, Any]]:
    """
    Накапливает по каждому (канал, бэнд) значения всех триалов как независимые наблюдения.
    Возвращает trial_vecs[(ch, band)] = {'ids': [...], 's_ids': [...], 'vals': np.ndarray}
    """
    trial_vecs: Dict[Tuple[int, str], Dict[str, Any]] = defaultdict(
        lambda: {'ids': [], 's_ids': [], 'vals': []}
    )

    for rec in results:
        power = rec.get("power", None)
        s_id  = rec.get("s_id", None)
        t_id  = rec.get("t_id", None)
        if power is None:
            continue

        # (C,F,T) -> среднее по времени; иначе ожидаем (C,F)
        if hasattr(power, "ndim") and power.ndim == 3:
            vecs = power.mean(axis=2).astype(np.float32, copy=False)
        else:
            vecs = np.asarray(power, dtype=np.float32)
        if vecs.ndim != 2:
            continue

        try:
            sid_int = int(s_id) if s_id is not None else None
        except Exception:
            sid_int = None
        try:
            tid_int = int(t_id) if t_id is not None else None
        except Exception:
            tid_int = None

        max_ch = min(n_channels, vecs.shape[0])
        for ch in range(max_ch):
            for band_name, freq_idx in band_cols.items():
                if freq_idx.size == 0 or np.max(freq_idx) >= vecs.shape[1]:
                    continue
                band_val = float(np.nanmean(vecs[ch, freq_idx]))
                key = (ch, band_name)
                trial_vecs[key]['ids'].append(tid_int)
                trial_vecs[key]['s_ids'].append(sid_int)
                trial_vecs[key]['vals'].append(band_val)

    for key, d in trial_vecs.items():
        d['vals'] = np.asarray(d['vals'], dtype=float)

    return trial_vecs

def build_two_condition_vectors(
    condA_subj: Dict[Tuple[int, str], Dict[str, Any]],
    condB_subj: Dict[Tuple[int, str], Dict[str, Any]],
    condA_name: str,
    condB_name: str,
    n_channels: int,
    band_names: List[str],
) -> Dict[Tuple[int, str], Dict[str, Any]]:
    """
    Объединяет векторы двух условий в структуру {(ch, band): {...}}.
    """
    subject_vectors: Dict[Tuple[int, str], Dict[str, Any]] = {}

    for ch in range(n_channels):
        for band_name in band_names:
            key = (ch, band_name)

            A = condA_subj.get(key, {'ids': [], 's_ids': [], 'vals': np.array([], float)})
            B = condB_subj.get(key, {'ids': [], 's_ids': [], 'vals': np.array([], float)})

            x = np.asarray(A.get('vals', np.array([], float)), dtype=float)
            y = np.asarray(B.get('vals', np.array([], float)), dtype=float)

            subject_vectors[key] = {
                condA_name: x,
                condB_name: y,
                f"{condA_name}_ids": list(A.get('ids', [])),
                f"{condB_name}_ids": list(B.get('ids', [])),
                f"{condA_name}_s_ids": list(A.get('s_ids', [])),
                f"{condB_name}_s_ids": list(B.get('s_ids', [])),
            }

    return subject_vectors

# ==== эффекты и статистика ====
def hedges_g(x, y) -> float:
    """
    Hedges' g для двух независимых групп (с поправкой J).
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)
    if vx == 0 and vy == 0:
        return float(np.sign(np.nanmean(x) - np.nanmean(y)) * 0.0)
    sp2 = ((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2)
    if sp2 <= 0:
        return np.nan
    d = (np.nanmean(x) - np.nanmean(y)) / np.sqrt(sp2)
    J = 1 - 3 / (4 * (nx + ny) - 9) if (nx + ny) > 2 else 1.0
    return float(J * d)

def build_band_table_two_conditions(
    subject_vectors: Dict[Tuple[int, str], Dict[str, Any]],
    condA_name: str,
    condB_name: str,
    alpha: float = 0.05
) -> pd.DataFrame:
    """
    Для каждой пары (канал, бэнд):
      - кластер-устойчивый t-тест по испытуемому (если можно)
      - иначе Welch t-test
      - p-value, Hedges' g, оценка мощности
    """
    try:
        import statsmodels.api as sm
        _HAS_SM = True
    except Exception:
        _HAS_SM = False

    power_calc = TTestIndPower()
    rows = []

    for (ch, band_name), vecs in subject_vectors.items():
        x = np.asarray(vecs.get(condA_name, []), float)
        y = np.asarray(vecs.get(condB_name, []), float)
        nA, nB = len(x), len(y)
        if nA == 0 or nB == 0:
            continue

        sA = np.asarray(vecs.get(f"{condA_name}_s_ids", []))
        sB = np.asarray(vecs.get(f"{condB_name}_s_ids", []))

        # Welch по умолчанию
        try:
            t_stat, p_val = stats.ttest_ind(x, y, equal_var=False, nan_policy='omit')
        except Exception:
            t_stat, p_val = np.nan, np.nan

        g = hedges_g(x, y)
        used_cluster = False

        # Кластер-устойчивые СКО по испытуемому (если возможно)
        if _HAS_SM and (sA.size == nA) and (sB.size == nB):
            mA = pd.notna(sA); mB = pd.notna(sB)
            x2, y2 = x[mA], y[mB]
            sA2, sB2 = sA[mA], sB[mB]
            if (len(x2) >= 2 and len(y2) >= 2 and
                len(np.unique(sA2)) >= 2 and len(np.unique(sB2)) >= 2):
                try:
                    y_all = np.concatenate([x2, y2])
                    g_all = np.concatenate([np.zeros_like(x2), np.ones_like(y2)])
                    sid_all = np.concatenate([sA2, sB2])
                    X = sm.add_constant(g_all.astype(float))
                    res = sm.OLS(y_all, X, missing='drop').fit(
                        cov_type='cluster', cov_kwds={'groups': sid_all}
                    )
                    t_stat = float(res.tvalues[1])
                    p_val  = float(res.pvalues[1])
                    used_cluster = True

                    # мощность по средним на кластерах
                    dfA = pd.DataFrame({'sid': sA2, 'val': x2}).groupby('sid')['val'].mean()
                    dfB = pd.DataFrame({'sid': sB2, 'val': y2}).groupby('sid')['val'].mean()
                    nA_eff, nB_eff = len(dfA), len(dfB)
                    if nA_eff >= 2 and nB_eff >= 2:
                        g_eff = hedges_g(dfA.values, dfB.values)
                        ratio = nB_eff / max(nA_eff, 1)
                        power = power_calc.solve_power(effect_size=abs(g_eff),
                                                       nobs1=nA_eff, ratio=ratio,
                                                       alpha=alpha, alternative='two-sided')
                    else:
                        power = np.nan
                except Exception:
                    used_cluster = False

        if not used_cluster:
            try:
                ratio = nB / max(nA, 1)
                power = power_calc.solve_power(effect_size=abs(g),
                                               nobs1=nA, ratio=ratio,
                                               alpha=alpha, alternative='two-sided')
            except Exception:
                power = np.nan

        rows.append({
            'channel': ch,
            'band': band_name,
            f'n_{condA_name}': nA,
            f'n_{condB_name}': nB,
            f'mean_{condA_name}': float(np.nanmean(x)) if nA else np.nan,
            f'mean_{condB_name}': float(np.nanmean(y)) if nB else np.nan,
            f'delta_{condA_name}_minus_{condB_name}':
                float(np.nanmean(x) - np.nanmean(y)) if (nA and nB) else np.nan,
            't_stat': float(t_stat) if np.isfinite(t_stat) else np.nan,
            'p_value': float(p_val) if np.isfinite(p_val) else np.nan,
            'hedges_g': float(g) if np.isfinite(g) else np.nan,
            'power': float(power) if np.isfinite(power) else np.nan,
            'sig_alpha_0.05': bool((p_val <= 0.05) if np.isfinite(p_val) else False),
            'sig_alpha_0.01': bool((p_val <= 0.01) if np.isfinite(p_val) else False),
            'sig_and_power': bool(
                (p_val <= 0.05) and (power >= 0.8)
            ) if (np.isfinite(p_val) and np.isfinite(power)) else False,
            'clustered': used_cluster,
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['p_value', 'power'],
                            ascending=[True, True]).reset_index(drop=True)
    return df

def run_stats_between_groups(
    data_A_raw,
    data_B_raw,
    group_A_name: str,
    group_B_name: str,
    band_cols: Dict[str, np.ndarray],
    n_channels: int,
    alpha: float = 0.05,
):
    # нормализация формата
    data_A = normalize_results(data_A_raw)
    data_B = normalize_results(data_B_raw)

    # агрегация (по триалам)
    A_by_subj = aggregate_by_subject(data_A, band_cols=band_cols, n_channels=n_channels)
    B_by_subj = aggregate_by_subject(data_B, band_cols=band_cols, n_channels=n_channels)

    band_names = list(band_cols.keys())

    # объединяем векторы
    subj_vecs = build_two_condition_vectors(
        condA_subj=A_by_subj,
        condB_subj=B_by_subj,
        condA_name=group_A_name,
        condB_name=group_B_name,
        n_channels=n_channels,
        band_names=band_names,
    )

    # статистика
    stats_table = build_band_table_two_conditions(
        subject_vectors=subj_vecs,
        condA_name=group_A_name,
        condB_name=group_B_name,
        alpha=alpha,
    )

    return {
        'summary_table': stats_table,
        'meta': {'group_A': group_A_name, 'group_B': group_B_name}
    }

def _collapse_allowed_values(allowed_values: List[Any]):
    """
    Если группа — одна категория (например ["m"]) → вернуть "m".
    Если это бин (например [18,19,20,...]) — вернуть весь список.
    """
    if len(allowed_values) == 1:
        return allowed_values[0]
    return allowed_values

# ==== ВАЖНО: НОВАЯ filter_group (всегда зовёт filter_spectras, в т.ч. c day_time) ====
def filter_group(
    npz_path: str,
    feature_name: str,
    allowed_values: List[Any],
    extra_kwargs: dict,
):
    """
    Всегда вызывает filter_spectras.
    Для day_time — ПЕРЕДАЁМ day_time (и meta-путь).
    Кэш _all_trials не используем. Пустые группы возвращают [].
    """
    if extra_kwargs is None or "day_time_meta_path" not in extra_kwargs:
        raise RuntimeError("Нужен extra_kwargs['day_time_meta_path'] (путь к Excel).")

    val = _collapse_allowed_values(allowed_values)
    kwargs = {
        "exec_spec_path": npz_path,
        "day_time_meta_path": extra_kwargs["day_time_meta_path"],
    }

    if feature_name == "day_time":
        kwargs["day_time"] = val
    elif feature_name == "stim_type":
        kwargs["stim_type"] = val
    elif feature_name == "stim_label":
        kwargs["stim_label"] = val
    elif feature_name == "gender":
        kwargs["gender"] = val
    elif feature_name == "age":
        kwargs["age"] = val
    elif feature_name == "handiness":
        kwargs["handiness"] = val
    else:
        raise ValueError(f"Unknown feature: {feature_name}")

    # гасим overflow/invalid внутри вызова
    with _np_ignore_overflow():
        out = filter_spectras(**kwargs)

    return out if out is not None else []

# ==== подготовка сессии (оставляем как было; кэш не используется далее, но не мешает) ====
def prepare_session_config_for_npz(
    npz_path: str,
    bands: Dict[str, Tuple[float, float]] = None,
    extra_kwargs: dict = None,
):
    if bands is None:
        bands = DEFAULT_BANDS.copy()
    if extra_kwargs is None:
        extra_kwargs = {}

    meta_path = extra_kwargs.get("day_time_meta_path", None)
    if meta_path is None:
        raise RuntimeError(
            "day_time_meta_path обязателен: передай extra_kwargs={'day_time_meta_path': '...xlsx'}"
        )

    base_kwargs = {
        "exec_spec_path": npz_path,
        "day_time_meta_path": meta_path,
    }

    # берём все триалы просто для оценки размерностей
    with _np_ignore_overflow():
        all_trials_raw = filter_spectras(**base_kwargs)

    all_trials_norm = normalize_results(all_trials_raw or [])

    # найдём пример с power
    example_power = None
    for rec in all_trials_norm:
        if "power" in rec and rec["power"] is not None:
            example_power = rec["power"]
            break
    if example_power is None:
        raise RuntimeError("Не удалось найти ни одного trial с 'power' в этом npz (после filter_spectras).")

    # размерности
    if hasattr(example_power, "ndim") and example_power.ndim == 3:
        n_channels, n_freqs, _ = example_power.shape
    elif hasattr(example_power, "ndim") and example_power.ndim == 2:
        n_channels, n_freqs = example_power.shape
    else:
        raise ValueError(f"Неожиданная форма 'power': {getattr(example_power,'shape',None)}")

    # сетка частот
    freqs = np.linspace(2, 40, n_freqs)

    band_cols = make_band_cols_from_bands(bands, freqs)

    session_cfg = {
        "npz_path": npz_path,
        "n_channels": n_channels,
        "band_cols": band_cols,
        "bands": bands,
        "extra_kwargs": extra_kwargs,
        "_all_trials": all_trials_norm,  # не используется далее, оставлено на будущее
    }
    return session_cfg

# ==== НОВАЯ analyze_single_feature (без кэша, всегда через filter_spectras) ====
def analyze_single_feature(
    session_cfg: dict,
    feature_name: str,
    alpha: float = 0.05,
):
    """
    Для каждой группы фичи зовём filter_spectras (для day_time — с day_time),
    оставляем только непустые группы и считаем попарно.
    """
    npz_path     = session_cfg["npz_path"]
    band_cols    = session_cfg["band_cols"]
    n_channels   = session_cfg["n_channels"]
    extra_kwargs = session_cfg["extra_kwargs"]

    group_spec = FEATURE_GROUPS[feature_name]

    subsets = {}
    for group_label, allowed_values in group_spec.items():
        data = filter_group(
            npz_path=npz_path,
            feature_name=feature_name,
            allowed_values=allowed_values,
            extra_kwargs=extra_kwargs,
        )
        if data and len(data) > 0:
            subsets[group_label] = data

    if len(subsets) < 2:
        return {}

    results_for_feature = {}
    for g1, g2 in itertools.combinations(subsets.keys(), 2):
        data_A_raw = subsets[g1]
        data_B_raw = subsets[g2]
        if not data_A_raw or not data_B_raw:
            continue

        stats_result = run_stats_between_groups(
            data_A_raw=data_A_raw,
            data_B_raw=data_B_raw,
            group_A_name=g1,
            group_B_name=g2,
            band_cols=band_cols,
            n_channels=n_channels,
            alpha=alpha,
        )
        df = stats_result.get("summary_table")
        if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
            results_for_feature[(g1, g2)] = stats_result

    return results_for_feature

# ==== НОВАЯ analyze_feature_pair (без кэша, всегда через filter_spectras) ====
def analyze_feature_pair(
    session_cfg: dict,
    feature_A: str,
    feature_B: str,
    alpha: float = 0.05,
):
    """
    Строим группы по обеим фичам через filter_spectras (для day_time — с day_time),
    пропускаем пустые и считаем все непустые комбинации.
    """
    npz_path     = session_cfg["npz_path"]
    band_cols    = session_cfg["band_cols"]
    n_channels   = session_cfg["n_channels"]
    extra_kwargs = session_cfg["extra_kwargs"]

    groups_A = FEATURE_GROUPS[feature_A]
    groups_B = FEATURE_GROUPS[feature_B]

    subsets_A, subsets_B = {}, {}

    for label_A, allowed_values_A in groups_A.items():
        dataA = filter_group(
            npz_path=npz_path,
            feature_name=feature_A,
            allowed_values=allowed_values_A,
            extra_kwargs=extra_kwargs,
        )
        if dataA and len(dataA) > 0:
            subsets_A[label_A] = dataA

    for label_B, allowed_values_B in groups_B.items():
        dataB = filter_group(
            npz_path=npz_path,
            feature_name=feature_B,
            allowed_values=allowed_values_B,
            extra_kwargs=extra_kwargs,
        )
        if dataB and len(dataB) > 0:
            subsets_B[label_B] = dataB

    if len(subsets_A) == 0 or len(subsets_B) == 0:
        return {}

    results_for_pair = {}
    for label_A, data_A_raw in subsets_A.items():
        for label_B, data_B_raw in subsets_B.items():
            if not data_A_raw or not data_B_raw:
                continue

            stats_result = run_stats_between_groups(
                data_A_raw=data_A_raw,
                data_B_raw=data_B_raw,
                group_A_name=label_A,
                group_B_name=label_B,
                band_cols=band_cols,
                n_channels=n_channels,
                alpha=alpha,
            )
            df = stats_result.get("summary_table")
            if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
                results_for_pair[(label_A, label_B)] = stats_result

    return results_for_pair

# ==== основной раннер ====
def run_full_analysis_for_npz(
    npz_path: str,
    alpha: float = 0.05,
    feature_list_single=None,
    feature_pairs=None,
    extra_kwargs: dict = None,
    bands: Dict[str, Tuple[float, float]] = None,
):
    if extra_kwargs is None or "day_time_meta_path" not in extra_kwargs:
        raise RuntimeError(
            "run_full_analysis_for_npz: передай extra_kwargs={'day_time_meta_path': '...xlsx'}"
        )

    session_cfg = prepare_session_config_for_npz(
        npz_path=npz_path,
        bands=bands,
        extra_kwargs=extra_kwargs,
    )

    if feature_list_single is None:
        feature_list_single = list(FEATURE_GROUPS.keys())
    if feature_pairs is None:
        feature_pairs = list(itertools.combinations(FEATURE_GROUPS.keys(), 2))

    final_results = {"single_feature": {}, "pair_feature": {}}

    # одиночные фичи
    for feat in feature_list_single:
        final_results["single_feature"][feat] = analyze_single_feature(
            session_cfg=session_cfg,
            feature_name=feat,
            alpha=alpha,
        )

    # пары фичей
    for (featA, featB) in feature_pairs:
        final_results["pair_feature"][(featA, featB)] = analyze_feature_pair(
            session_cfg=session_cfg,
            feature_A=featA,
            feature_B=featB,
            alpha=alpha,
        )

    return final_results


In [20]:
# --- пути и параметры ---
npz_path  = r'./Generated/Spectrums/psds_array_morlet.npz'
meta_path = r'./Supplementary/Experiment_Metadata.xlsx'

bands = {
    'Delta': (1, 4),
    'Tetta': (4, 7),
    'Alpha': (7, 13),
    'Beta':  (13, 30),
}

extra_kwargs = {
    "day_time_meta_path": meta_path
}

# --- запуск ---
results = run_full_analysis_for_npz(
    npz_path=npz_path,
    alpha=0.05,
    extra_kwargs=extra_kwargs,
    bands=bands,
    feature_list_single=None,   # None => все одиночные фичи
    feature_pairs=None,         # None => все пары фичей
)

print("Готово: посчитаны таблицы для существующих и непустых групп.")


Готово: посчитаны таблицы для существующих и непустых групп.


In [21]:
from IPython.display import display
import pandas as pd

feat = "gender"
pair = ("m", "f")

block = results.get("single_feature", {}).get(feat, {})
if pair in block:
    df = block[pair].get("summary_table")
    if isinstance(df, pd.DataFrame) and not df.empty:
        print(f"{feat.upper()}: {pair[0]} vs {pair[1]} • top по p-value")
        display(df.sort_values("p_value").head(10))

        interesting = df[(df["p_value"] <= 0.05) & (df["power"] >= 0.8)]\
                       .sort_values(["p_value","power"], ascending=[True, False])
        if not interesting.empty:
            print(f"{feat.upper()}: значимые (p<=0.05 & power>=0.8)")
            display(interesting.head(20))
        else:
            print("Значимых строк нет по заданным порогам.")
    else:
        print("Таблица есть логически, но пуста (недостаточно данных).")
else:
    print(f"Нет сравнения {pair} в results['single_feature']['{feat}'] "
          f"(группы могли быть пустыми и были пропущены).")


GENDER: m vs f • top по p-value


,channel,band,n_m,n_f,mean_m,mean_f,delta_m_minus_f,t_stat,p_value,hedges_g,power,sig_alpha_0.05,sig_alpha_0.01,sig_and_power,clustered
0,39,Beta,16,15,0.039274,0.081607,-0.042332,3.170358,0.001523,-1.219303,0.721233,True,True,False,True
1,58,Beta,16,15,0.046096,0.107414,-0.061319,3.167440,0.001538,-1.289985,0.705409,True,True,False,True
2,35,Tetta,16,15,0.147817,0.292903,-0.145086,2.790919,0.005256,-0.881503,0.669984,True,True,False,True
3,28,Beta,16,15,0.044466,0.104961,-0.060494,2.710305,0.006722,-1.240163,0.680423,True,True,False,True
4,8,Beta,16,15,0.069899,0.196037,-0.126138,2.621877,0.008745,-1.157393,0.547011,True,True,False,True
5,60,Beta,16,15,0.040080,0.115407,-0.075327,2.561009,0.010437,-1.180479,0.564988,True,False,False,True
6,56,Beta,16,15,0.036147,0.061226,-0.025079,2.527856,0.011476,-0.987354,0.490670,True,False,False,True
7,61,Beta,16,15,0.040416,0.066429,-0.026013,2.471360,0.013460,-1.165992,0.549851,True,False,False,True
8,5,Beta,16,15,0.056774,0.094222,-0.037449,2.461255,0.013845,-1.019987,0.501305,True,False,False,True
9,48,Beta,16,15,0.058988,0.087479,-0.028491,2.442529,0.014585,-0.983336,0.501201,True,False,False,True


Значимых строк нет по заданным порогам.


In [22]:
from IPython.display import display
import pandas as pd

featA, featB = "day_time", "stim_type"
pair_key = (featA, featB)
groups   = ("Day", "r")  # что показать

pair_block = results.get("pair_feature", {}).get(pair_key, {})
if groups in pair_block:
    df = pair_block[groups].get("summary_table")
    if isinstance(df, pd.DataFrame) and not df.empty:
        print(f"PAIR: {featA}={groups[0]} vs {featB}={groups[1]} • top по p-value")
        display(df.sort_values("p_value").head(10))

        interesting = df[(df["p_value"] <= 0.05) & (df["power"] >= 0.8)]\
                       .sort_values(["p_value","power"], ascending=[True, False])
        if not interesting.empty:
            print("PAIR: значимые (p<=0.05 & power>=0.8)")
            display(interesting.head(20))
        else:
            print("Значимых строк нет по заданным порогам.")
    else:
        print("Таблица есть логически, но пуста (недостаточно данных).")
else:
    print(f"Нет сравнения {groups} в pair_feature[{pair_key}] "
          f"(одна из групп могла отсутствовать в данных и была пропущена).")


Нет сравнения ('Day', 'r') в pair_feature[('day_time', 'stim_type')] (одна из групп могла отсутствовать в данных и была пропущена).


In [23]:
feat = "day_time"
pair = ("Day", "Evening")

block = results.get("single_feature", {}).get(feat, {})
if pair in block:
    df = block[pair].get("summary_table")
    if isinstance(df, pd.DataFrame) and not df.empty:
        print(f"{feat.upper()}: {pair[0]} vs {pair[1]} • top по p-value")
        display(df.sort_values("p_value").head(10))

        interesting = df[(df["p_value"] <= 0.05) & (df["power"] >= 0.8)]\
                       .sort_values(["p_value","power"], ascending=[True, False])
        if not interesting.empty:
            print(f"{feat.upper()}: значимые (p<=0.05 & power>=0.8)")
            display(interesting.head(20))
        else:
            print("Значимых строк нет по заданным порогам.")
    else:
        print("Таблица есть логически, но пуста (недостаточно данных).")
else:
    print(f"Нет сравнения {pair} в results['single_feature']['{feat}'] "
          f"(группы могли быть пустыми и были пропущены).")

DAY_TIME: Day vs Evening • top по p-value


,channel,band,n_Day,n_Evening,mean_Day,mean_Evening,delta_Day_minus_Evening,t_stat,p_value,hedges_g,power,sig_alpha_0.05,sig_alpha_0.01,sig_and_power,clustered
0,1,Beta,17,14,0.072493,0.036717,0.035777,-2.885669,0.003906,0.873760,0.534583,True,True,False,True
1,15,Beta,17,14,0.085475,0.058647,0.026828,-2.323932,0.020129,0.926293,0.535515,True,False,False,True
2,39,Beta,17,14,0.074711,0.041600,0.033112,-2.304192,0.021212,0.883776,0.431623,True,False,False,True
3,9,Beta,17,14,0.183673,0.068118,0.115555,-2.293753,0.021805,0.651398,0.387551,True,False,False,True
4,20,Beta,17,14,0.105969,0.068724,0.037245,-2.187983,0.028671,0.695897,0.403859,True,False,False,True
5,61,Beta,17,14,0.063045,0.040809,0.022236,-2.180162,0.029245,0.948003,0.394497,True,False,False,True
6,28,Beta,17,14,0.095189,0.047690,0.047499,-2.152159,0.031385,0.900905,0.412706,True,False,False,True
7,8,Beta,17,14,0.176425,0.075693,0.100732,-2.119385,0.034058,0.866205,0.338375,True,False,False,True
8,10,Beta,17,14,0.075780,0.047539,0.028241,-2.035907,0.041760,0.728774,0.314018,True,False,False,True
9,60,Beta,17,14,0.102106,0.045471,0.056636,-1.945313,0.051737,0.819734,0.309183,False,False,False,True


Значимых строк нет по заданным порогам.
